In [2]:
import pandas as pd

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(X_train.shape, X_test.shape)

(2400, 26) (600, 26)


In [3]:
from sklearn.preprocessing import StandardScaler

# Logistic Regression distance/weights pe based hai, isliye agar features ka
# scale bahut alag ho (jaise land_area 5-400 vs stakeholder_responsiveness 0-1),
# toh bade-scale wale features unfairly zyada dominate kar sakte hain.
# StandardScaler har column ko "mean=0, std=1" pe convert kar deta hai -- fair banata hai.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled. Example before/after (land_area_hectares):")
print("Before:", X_train["land_area_hectares"].iloc[0])
print("After :", X_train_scaled[0][list(X_train.columns).index("land_area_hectares")])

Scaled. Example before/after (land_area_hectares):
Before: 19.8
After : -0.9384789976007384


In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained!")

Model trained!


In [5]:
y_pred = model.predict(X_test_scaled)           # 0 ya 1 (final decision)
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # actual probability (0-1)

print("Sample predictions:", y_pred[:10])
print("Sample probabilities:", y_proba[:10].round(2))

Sample predictions: [1 0 1 0 1 0 0 1 0 0]
Sample probabilities: [0.89 0.08 0.52 0.15 0.77 0.05 0.04 0.86 0.13 0.01]


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("F1       :", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_proba), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

Precision: 0.686
Recall   : 0.78
F1       : 0.73
ROC-AUC  : 0.864

Confusion matrix:
 [[292  81]
 [ 50 177]]


In [ ]:
# Abhi tak model sirf tumhari notebook ki memory (RAM) mein hai — jaise hi notebook band karogi ya kernel restart hoga, ye sab seekha hua khatam ho jaayega, aur dobara train karna padega.
# joblib.dump() model ke saare seekhe hue weights ko ek file (.pkl) mein permanently save kar deta hai — jaise ek "save game" file. Baad mein sirf ek line se wapas load kar sakte ho:

import joblib
import os

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/baseline_logreg.pkl")
joblib.dump(scaler, "models/scaler.pkl")   # scaler bhi save karo -- prediction time pe same scaling chahiye hogi

print("Saved: models/baseline_logreg.pkl")

Saved: models/baseline_logreg.pkl
